In [9]:
import os
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp

from mediapipe.tasks import python
from mediapipe.tasks.python import vision



In [12]:
def build_ugly_recording_dataset(
    video_folder,
    rating_path,
    model_path,
    output_folder,
    c=30
):
    os.makedirs(output_folder, exist_ok=True)

    if not os.path.exists(rating_path):
        raise FileNotFoundError(f"Could not find rating file: {rating_path}")

    ratings_df = pd.read_csv(rating_path)
    ratings_dict = dict(zip(ratings_df["video"], ratings_df["rating"]))

    video_extensions = [".mp4", ".mov", ".avi", ".mkv"]

    JOINT_ORDER = [
        "head",
        "left_shoulder", "left_elbow",
        "right_shoulder", "right_elbow",
        "left_hand", "right_hand",
        "left_hip", "right_hip",
        "left_knee", "right_knee",
        "left_foot", "right_foot"
    ]

    LANDMARK_INDEX = {
        "head": 0,
        "left_shoulder": 11,
        "right_shoulder": 12,
        "left_elbow": 13,
        "right_elbow": 14,
        "left_hand": 15,
        "right_hand": 16,
        "left_hip": 23,
        "right_hip": 24,
        "left_knee": 25,
        "right_knee": 26,
        "left_foot": 27,
        "right_foot": 28,
    }

    def empty_frame_features():
        values = []

        for joint in JOINT_ORDER:
            values += [0.0, 0.0, 0.0, 0.0, 0.0]

        values.append(0.0)  # pose_score
        return values

    def extract_frame_features(results):
        if not results.pose_landmarks:
            return empty_frame_features()

        landmarks = results.pose_landmarks[0]

        values = []
        visibility_values = []
        presence_values = []

        for joint in JOINT_ORDER:
            idx = LANDMARK_INDEX[joint]
            lm = landmarks[idx]

            visibility = getattr(lm, "visibility", 0.0)
            presence = getattr(lm, "presence", 0.0)

            values += [
                lm.x,
                lm.y,
                lm.z,
                visibility,
                presence
            ]

            visibility_values.append(visibility)
            presence_values.append(presence)

        pose_score = np.mean(visibility_values + presence_values)
        values.append(pose_score)

        return values

    columns = []

    for frame_idx in range(c):
        for joint in JOINT_ORDER:
            columns += [
                f"frame{frame_idx}_{joint}_x",
                f"frame{frame_idx}_{joint}_y",
                f"frame{frame_idx}_{joint}_z",
                f"frame{frame_idx}_{joint}_visibility",
                f"frame{frame_idx}_{joint}_presence"
            ]

        columns.append(f"frame{frame_idx}_pose_score")

    columns.append("target")

    base_options = python.BaseOptions(model_asset_path=model_path)

    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5
    )

    for file_name in os.listdir(video_folder):

        if not any(file_name.lower().endswith(ext) for ext in video_extensions):
            continue

        video_id = os.path.splitext(file_name)[0]

        if video_id not in ratings_dict:
            print(f"Skipping {file_name}: no rating found")
            continue

        target = ratings_dict[video_id]
        video_path = os.path.join(video_folder, file_name)

        print(f"Processing: {file_name} | target={target}")

        row = []

        # Important: new landmarker for every video
        with vision.PoseLandmarker.create_from_options(options) as landmarker:

            cap = cv2.VideoCapture(video_path)

            if not cap.isOpened():
                print(f"Skipping {file_name}: could not open video")
                continue

            fps = cap.get(cv2.CAP_PROP_FPS)

            if fps <= 0:
                fps = 30

            frame_idx = 0

            while frame_idx < c:

                ret, frame = cap.read()

                if not ret:
                    row += empty_frame_features()
                    frame_idx += 1
                    continue

                image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

                mp_image = mp.Image(
                    image_format=mp.ImageFormat.SRGB,
                    data=image_rgb
                )

                timestamp_ms = int((frame_idx / fps) * 1000)

                results = landmarker.detect_for_video(
                    mp_image,
                    timestamp_ms
                )

                row += extract_frame_features(results)

                frame_idx += 1

            cap.release()

        row.append(target)

        output_df = pd.DataFrame([row], columns=columns)

        output_path = os.path.join(
            output_folder,
            f"{video_id}.csv"
        )

        output_df.to_csv(output_path, index=False)

        print(f"Saved: {output_path}")

    print("\nDone.")

In [24]:
#video_folder="../../../all_videos"

video_folder="../../../own_videos"


rating_path="../../MainProject/Assignment14/video_quality_rating.csv"
model_path="../data/pose_landmarker.task"
output_folder="../../MainProject/data/mediapipe_ugly_recordings"

build_ugly_recording_dataset(
    video_folder=video_folder,
    rating_path=rating_path,
    model_path=model_path,
    output_folder=output_folder,
    c=30
)

Processing: C1.mov | target=1


I0000 00:00:1779052139.635411 4007909 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
W0000 00:00:1779052139.701918 4007911 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779052139.716343 4007916 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Saved: ../../MainProject/data/mediapipe_ugly_recordings/C1.csv
Processing: C12.mov | target=1


I0000 00:00:1779052141.130069 4007962 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
W0000 00:00:1779052141.188197 4007966 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779052141.198424 4007970 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Saved: ../../MainProject/data/mediapipe_ugly_recordings/C12.csv
Processing: C13.mov | target=1


...

Saved: ../../MainProject/data/mediapipe_ugly_recordings/C20.csv
Processing: C22.mov | target=1


I0000 00:00:1779052175.928376 4009333 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
W0000 00:00:1779052175.990556 4009337 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779052175.999260 4009337 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Saved: ../../MainProject/data/mediapipe_ugly_recordings/C22.csv
Processing: C23.mov | target=1


I0000 00:00:1779052177.478021 4009385 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M2
W0000 00:00:1779052177.538837 4009388 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779052177.547829 4009388 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Saved: ../../MainProject/data/mediapipe_ugly_recordings/C23.csv

Done.


In [23]:
df = pd.read_csv("../../MainProject/data/mediapipe_ugly_recordings/A1.csv")

y = df["target"].values
X_flat = df.drop(columns=["target"]).values

n_frames = 30
n_features = 66

X = X_flat.reshape(-1, n_frames, n_features)
print(X.shape)

(1, 30, 66)
